In [2]:
!pip -q install transformers pandas

GPU and Drive

In [3]:
import os, json, numpy as np, pandas as pd, torch
assert torch.cuda.is_available(), "GPU missing: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))
from google.colab import drive; drive.mount("/content/drive")
D = "/content/drive/MyDrive/spoilfree/"
print(os.listdir(D))

Tesla T4
Mounted at /content/drive
['train.csv', 'val.csv', 'test._heldout.csv', 'eval_handwritten_107.csv']


Settings

In [4]:
teacher="bhavyagiri/roberta-base-finetuned-imdb-spoilers"
student="roberta-base"
maxlen=160
keep_pos=0.35 #keep top 35% according to teacher
keep_neg=0.70 #keep 70% of the least spoiler content
epochs=3
bs=32
lr=2e-5
seed=1

try:
    test_file=D+"test_heldout.csv"
    if not os.path.exists(test_file):
        test_file="/content/test._heldout.csv"
    if not os.path.exists(test_file):
        raise FileNotFoundError("test._heldout.csv not found")
    print("Using:", test_file)
except FileNotFoundError as e:
    print(f"File not found: {e}")

torch.manual_seed(seed)
np.random.seed(seed)

File not found: test._heldout.csv not found


**Helper Functions**

In [5]:
from sys import thread_info
from sklearn.metrics import roc_auc_score, average_precision_score

def clean_split(d,keep_posnew=None,keep_negnew=None,seed=1):
  keep_posnew=keep_pos if keep_posnew is None else keep_posnew
  keep_negnew=keep_neg if keep_negnew is None else keep_negnew
  pos,neg=d[d.label==1],d[d.label==0]
  pos=pos[pos.t1>=pos.t1.quantile(1-keep_posnew)]
  neg=neg[neg.t1>=neg.t1.quantile(keep_negnew)]
  k=min(len(pos),len(neg))
  out=pd.concat(pos.sample(k,random_state=seed),neg.sample(k,random_state=seed))
  return out.sample(frac=1,random_state=seed).reset_index(drop=True)[["text","label","t1"]]


def sweep(y,p,name,ths=(0.3,0.5,0.7,0.8,0.9,0.95)):
  y=np.asarray(y)
  p=np.asarray(p)
  print(f"{name} AUC: {roc_auc_score(y,p):.3f} AP {average_precision_score(y,p):.3f}")
  print(f"thr precision recall falseblur missed")
  for ts in ths:
    pr=p>=ts
    tp=int((pr & (y==1)).sum())
    fp=int((pr & (y==0)).sum())       #confusion matrix metric
    fn=int(((~pr) & (y==1)).sum())
    tn=int(((~pr) & (y==0)).sum())
    print(f"{ts:<6} {tp / max(tp + fp, 1):<10.3f} {tp / max(tp + fn, 1):<7.3f} {fp:<10} {fn}")


def pick_threshold(y,p,min_precision=0.95):
  #lowest precision that is >=min_precision if no threshold reaches then None
  y=np.asarray(y)
  p=np.asarray(p)
  for ts in np.arange(0.05,1.0,0.01):
    pr=p>=ts
    if pr.sum() and (pr&(y==1)).sum()/pr.sum()>=min_precision:
      return round(float(ts),2)
  return None




**Score from the teacher**

In [6]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
ttok=AutoTokenizer.from_pretrained(teacher)    #loads tokenizer to convert text into model readable tokens
teacher=AutoModelForSequenceClassification.from_pretrained(teacher).half().cuda().eval()  #loads pretrained teacher model
                                                                                            #then uses half precision to reduce memory usage,
                                                                                            #moves model to gpu and then enables evaluation mode
                                                                                            #disabling training spiecific behaviour like a dropout

@torch.no_grad()  #disables gradient calculations: saves memory and computation
def score_with(model,tokenizer,texts,bs=128):
  #processes upto 128 texts per batch


  lengths=[len(t) for t in texts]            #calculates the length of each text
  order=np.argsort(lengths)                    #sorts indices by text length in ascending order
  out=np.zeros(len(texts),dtype=np.float32)             #creates an array to store spoiler scores

  for i in range(0,len(texts),bs):                  #iterates through texts in batches of size bs
    j=order[i:i+bs]                                #gets sorted indices for the current batch
    batch_texts=[texts[k] for k in j]                     #retrieves texts using the sorted indices

    x=tokenizer(                             #converts batch texts into model-readable tokens
        batch_texts,                         #passes the current batch of texts
        truncation=True,                         #truncates texts exceeding the token limit
        max_length=maxlen,                     #sets the maximum number of tokens per text
        padding=True,                         #pads shorter sequences to match the longest sequence in the batch
        return_tensors="pt"                     #returns tokenized data as pytorch tensors
    )

    x=x.to("cuda")       #moves tokenized tensors to the gpu
    logits=model(**x).logits      #gets raw prediction scores from the model
    logits=logits.float()          #converts logits to 32-bit floating-point values

    probabilities=torch.softmax(logits,dim=1)          #converts logits into class probabilities
    spoiler_scores=probabilities[:,1].cpu().numpy()       #extracts class 1 probabilities and converts them to a numpy array
    out[j]=spoiler_scores                                 #stores scores at their original text indices

  return out
for n, f in [("val","val.csv"), ("test",test_file),("train","train.csv")]:  #iterates through validation, test, and training datasets
  out=D+f"{n}_scored.csv"       #creates the output file path
  if os.path.exists(out):
    print(f"{n} already scored")
    continue                                                            #skips to the next dataset

  d=pd.read_csv(D+f)
  d["t1"]=score_with(teacher,ttok,d.text.tolist())    #calculates teacher spoiler probabilities for each text
  d.to_csv(out,index=False)
  print(f"{n} scored {len(d)}")

del teacher        #deletes the teacher model variable
torch.cuda.empty_cache()


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

val scored 10944


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/spoilfree//content/test._heldout.csv'